# Search-13 — Groupes automatiques : le problème du mot par automates (Rust / std)

**Série Search · Part 1 — Fondations** · Jumeau Rust de [Search-13 — Groupes automatiques](Search-13-AutomaticGroups-WordProblem.ipynb) — noyau `rust` (evcxr), **bibliothèque standard seule**

[← Retour à la série](../README.md) · Jumeau Python : [Search-13 — Groupes automatiques](Search-13-AutomaticGroups-WordProblem.ipynb) · Précédent : [Search-12a — Composer des regards](Search-12a-Composer-Regards.ipynb) · Suivant : [Search-13a — Traverser un mur](Search-13a-Traverser-Murs-Certifies.ipynb)

Quand la structure du groupe donne l'automate, *décider* d'une propriété d'un mot devient un test
d'appartenance à un langage **régulier** — linéaire en la longueur du mot — au lieu d'une **exploration**
du graphe de Cayley. Ce carnet rend calculable cette frontière, sur les deux groupes types :
`Z^2` (croissance polynômiale) et `F_2` (croissance exponentielle).

> Ce jumeau rejoue la même expérience en Rust, sous le noyau `rust` (evcxr) : **aucune crate**, aucun
> `:dep`, aucune dépendance hors de la bibliothèque standard. Les sorties sont **identiques** au
> caractère près à celles du jumeau Python — la page de spec (#28630) en donne les témoins.
>
> Provenance : Epstein, Cannon, Holt, Levy, Paterson, Thurston, *Word Processing in Groups* (1992) ;
> structure automatique de Gromov.

## Enjeu — la heuristique ultime

Les carnets frères de la série posent le contraste exploration :

- **Search-02** (non informé) explore par **BFS** — il *parcourt* l'espace d'états ;
- **Search-03** (informé) explore par **A\*** — une heuristique *guide* le parcours, mais on explore toujours.

Ce carnet ajoute la couche qui manque : **quand le groupe lui-même fournit l'automate**, il n'y a plus
d'exploration. La structure algébrique *est* la heuristique, et elle est complète : elle transforme
la question « ce mot est-il géodésique ? » en une lecture gauche-droite d'un automate fini, en **O(|mot|)**.

## 1. Le cadre — deux groupes, un alphabet de 4 lettres

On travaille sur l'alphabet `{a, a⁻¹, b, b⁻¹}` (deux générateurs et leurs inverses).

- **`Z^2`** (libre abélien de rang 2) : un mot vaut l'identité si la *somme des exposants* de `a` est 0
  ET celle de `b` est 0. Le groupe est commutatif : `a b a⁻¹ b⁻¹ = 1`.
- **`F_2`** (groupe libre de rang 2) : un mot se réduit par *annulation libre* — on efface chaque lettre
  adjacente à son inverse. `a b a⁻¹` ne se réduit pas (le `b` bloque), mais `a a⁻¹` s'annule.

In [2]:
// Alphabet : deux generateurs et leurs inverses. Un mot = LISTE de tokens.

#[allow(dead_code)]
fn inv(tok: &str) -> &'static str {
    match tok {
        "a" => "a-1",
        "a-1" => "a",
        "b" => "b-1",
        _ => "b",
    }
}

#[allow(dead_code)]
fn word(s: &str) -> Vec<String> {
    let chars: Vec<char> = s.chars().collect();
    let mut toks: Vec<String> = Vec::new();
    let mut i = 0usize;
    while i < chars.len() {
        if i + 2 < chars.len() && chars[i + 1] == '-' && chars[i + 2] == '1' {
            let mut tok = String::new();
            tok.push(chars[i]);
            tok.push_str("-1");
            toks.push(tok);
            i += 3;
        } else {
            toks.push(chars[i].to_string());
            i += 1;
        }
    }
    toks
}

#[allow(dead_code)]
fn eval_z2(w: &[String]) -> (i64, i64) {
    let mut x: i64 = 0;
    let mut y: i64 = 0;
    for l in w {
        match l.as_str() {
            "a" => x += 1,
            "a-1" => x -= 1,
            "b" => y += 1,
            "b-1" => y -= 1,
            _ => {}
        }
    }
    (x, y)
}

#[allow(dead_code)]
fn reduce_f2(w: &[String]) -> Vec<String> {
    let mut st: Vec<String> = Vec::new();
    for l in w {
        let cancels = match st.last() {
            Some(last) => last.as_str() == inv(l),
            None => false,
        };
        if cancels {
            st.pop();
        } else {
            st.push(l.clone());
        }
    }
    st
}

#[allow(dead_code)]
fn py_tuple_repr(p: (i64, i64)) -> String {
    format!("({}, {})", p.0, p.1)
}

#[allow(dead_code)]
fn py_list_repr(items: &[String]) -> String {
    let inner: Vec<String> = items.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", inner.join(", "))
}

// Controles de base
assert_eq!(eval_z2(&word("aba-1b-1")), (0, 0)); // a b a^-1 b^-1 = 1 dans Z^2 (commutatif)
assert!(reduce_f2(&word("aa-1")).is_empty()); // annulation libre
assert_eq!(reduce_f2(&word("abb-1")), word("a")); // b b^-1 s'annule, reste 'a'
println!(
    "Z^2 : a b a^-1 b^-1 vaut {}",
    py_tuple_repr(eval_z2(&word("aba-1b-1")))
);
println!(
    "F_2 : 'abb-1' se réduit en {}",
    py_list_repr(&reduce_f2(&word("abb-1")))
);


Z^2 : a b a^-1 b^-1 vaut (0, 0)


F_2 : 'abb-1' se réduit en ['a']


### Lecture du résultat

Dans `Z^2` le mot `a b a⁻¹ b⁻¹` vaut l'identité *par commutativité* (la somme des exposants est (0,0)),
alors que dans `F_2` le même enchaînement ne se réduit **pas** — la lettre `b` centrale bloque l'annulation
`a a⁻¹`. C'est toute la différence entre un groupe commutatif et un groupe libre : la *géométrie* des mots
n'y est pas du tout la même, et c'est précisément elle que les automates vont capturer.

## 2. Le problème du mot par automates — l'accepteur **construit**

Le langage des **mots géodésiques** (librement réduits) de `Z^2` est **régulier** : on le reconnaît par un
automate fini, et on le *construit* ici — l'état de l'automate n'est rien d'autre que **la dernière lettre
lue** (plus l'état de départ et un état mort pour les mots qui annulent).

In [3]:
// DFA CONSTRUIT des mots geodesiques (librement reduits) de Z^2.
// Etats : S (depart) + un etat par derniere lettre + D (etat mort, mot qui s'annule).

#[allow(dead_code)]
fn delta(etat: &str, lettre: &str) -> &'static str {
    match (etat, lettre) {
        ("S", "a") => "a",
        ("S", "a-1") => "a-1",
        ("S", "b") => "b",
        ("S", "b-1") => "b-1",
        ("a", "a") => "a",
        ("a", "a-1") => "D",
        ("a", "b") => "b",
        ("a", "b-1") => "b-1",
        ("a-1", "a") => "D",
        ("a-1", "a-1") => "a-1",
        ("a-1", "b") => "b",
        ("a-1", "b-1") => "b-1",
        ("b", "a") => "a",
        ("b", "a-1") => "a-1",
        ("b", "b") => "b",
        ("b", "b-1") => "D",
        ("b-1", "a") => "a",
        ("b-1", "a-1") => "a-1",
        ("b-1", "b") => "D",
        ("b-1", "b-1") => "b-1",
        _ => "D",
    }
}

#[allow(dead_code)]
fn inv(tok: &str) -> &'static str {
    match tok {
        "a" => "a-1",
        "a-1" => "a",
        "b" => "b-1",
        _ => "b",
    }
}

#[allow(dead_code)]
fn word(s: &str) -> Vec<String> {
    let chars: Vec<char> = s.chars().collect();
    let mut toks: Vec<String> = Vec::new();
    let mut i = 0usize;
    while i < chars.len() {
        if i + 2 < chars.len() && chars[i + 1] == '-' && chars[i + 2] == '1' {
            let mut tok = String::new();
            tok.push(chars[i]);
            tok.push_str("-1");
            toks.push(tok);
            i += 3;
        } else {
            toks.push(chars[i].to_string());
            i += 1;
        }
    }
    toks
}

#[allow(dead_code)]
fn is_geodesic(w: &[String]) -> bool {
    let mut s = "S";
    for l in w {
        s = delta(s, l.as_str());
    }
    matches!(s, "S" | "a" | "a-1" | "b" | "b-1")
}

// --- Verification PAR LEURS PROPRIETES (assertees, pas supposees) ---
// (1) un mot reduit est accepte
assert!(is_geodesic(&word("aba")));
assert!(is_geodesic(&word("b")));
assert!(is_geodesic(&word(""))); // mot vide = geodesique de longueur 0
// (2) un mot avec annulation est rejete
assert!(!is_geodesic(&word("aa-1")));
assert!(!is_geodesic(&word("ab-1b"))); // b-1 puis b s'annulent
assert!(!is_geodesic(&word("ba-1a")));
// (3) fermeture par produit : w geodesique, w.g l'est sauf si g annule la derniere lettre de w
for wc in ["a", "ab", "aba", "b-1", "ab-1"] {
    let w = word(wc);
    for g in ["a", "a-1", "b", "b-1"] {
        let mut wg = w.clone();
        wg.push(g.to_string());
        let attendu = g != inv(w.last().unwrap());
        assert!(is_geodesic(&wg) == attendu, "cas {} {}", wc, g);
    }
}

println!("Accepteur Z^2 : 6 etats (S, a, a-1, b, b-1, D), 5 acceptants.");
println!("Toutes les proprietes sont verifiees par les assertions ci-dessus.");


Accepteur Z^2 : 6 etats (S, a, a-1, b, b-1, D), 5 acceptants.


Toutes les proprietes sont verifiees par les assertions ci-dessus.


### Lecture du résultat

L'automate a **6 états** — bornés, indépendants de la longueur du mot. Il décide en une seule passe
gauche-droite, en **O(|mot|)** et en **mémoire constante** (un seul état courant). Les trois propriétés
demandées par #25674 sont vérifiées par les assertions : un mot réduit est accepté, un mot qui annule est
rejeté, et l'ensemble des mots acceptés est *fermé par produit* — multiplier un mot géodésique par un
générateur le laisse géodésique, sauf quand ce générateur annule la dernière lettre.

## 3. Les cônes géodésiques — la régularité, et l'automate qu'elle donne

Le **cône géodésique** de `Z^2` est l'ensemble des mots géodésiques ; dire que son langage est *régulier*
c'est dire qu'un automate fini le reconnaît. L'automate ci-dessus **est** ce témoin : sa table de
transition est finie et effective, pas une affirmation.

In [4]:
// La table de transition FINIE de l'accepteur geodesique de Z^2 : la regularite, effective.

#[allow(dead_code)]
fn delta(etat: &str, lettre: &str) -> &'static str {
    match (etat, lettre) {
        ("S", "a") => "a",
        ("S", "a-1") => "a-1",
        ("S", "b") => "b",
        ("S", "b-1") => "b-1",
        ("a", "a") => "a",
        ("a", "a-1") => "D",
        ("a", "b") => "b",
        ("a", "b-1") => "b-1",
        ("a-1", "a") => "D",
        ("a-1", "a-1") => "a-1",
        ("a-1", "b") => "b",
        ("a-1", "b-1") => "b-1",
        ("b", "a") => "a",
        ("b", "a-1") => "a-1",
        ("b", "b") => "b",
        ("b", "b-1") => "D",
        ("b-1", "a") => "a",
        ("b-1", "a-1") => "a-1",
        ("b-1", "b") => "D",
        ("b-1", "b-1") => "b-1",
        _ => "D",
    }
}

#[allow(dead_code)]
fn rjust(s: &str, w: usize) -> String {
    let len = s.chars().count();
    if len >= w {
        s.to_string()
    } else {
        format!("{}{}", " ".repeat(w - len), s)
    }
}

let lettres = ["a", "a-1", "b", "b-1"];
let etats = ["S", "a", "a-1", "b", "b-1", "D"];
let finaux = ["S", "a", "a-1", "b", "b-1"];

let entete: Vec<String> = lettres.iter().map(|l| rjust(l, 4)).collect();
println!("  \\  | {}", entete.join(" | "));
println!("{}", "-".repeat(42));
for s in etats.iter().copied() {
    let row: Vec<String> = lettres.iter().map(|l| rjust(delta(s, l), 4)).collect();
    let marque = if finaux.contains(&s) {
        "   <== acceptant"
    } else if s == "D" {
        "   (etat mort)"
    } else {
        ""
    };
    println!("  {} | {}{}", rjust(s, 3), row.join("   "), marque);
}
println!(
    "\nNombre d'etats : {} (borné, indépendant de la longueur des mots).",
    etats.len()
);


  \  |    a |  a-1 |    b |  b-1


------------------------------------------


    S |    a    a-1      b    b-1   <== acceptant


    a |    a      D      b    b-1   <== acceptant


  a-1 |    D    a-1      b    b-1   <== acceptant


    b |    a    a-1      b      D   <== acceptant


  b-1 |    a    a-1      D    b-1   <== acceptant


    D |    D      D      D      D   (etat mort)


Nombre d'etats : 6 (borné, indépendant de la longueur des mots).


## 4. Le contraste qui compte — BFS (explorer) vs accepteur (décider)

Pour savoir si un mot de longueur `R` est géodésique, deux stratégies :

- **BFS** sur le graphe de Cayley de `Z^2` : on explore la boule de rayon `R`, qui compte **Θ(R²)**
  noeuds — on *explore* pour décider ;
- **accepteur** : on lit le mot en **O(|mot|) = O(R)**, *sans explorer*.

Le rapport *noeuds-BFS / états-accepteur* sur des sphères croissantes fait voir l'écart se creuser.

In [5]:
// Le BFS de la boule de rayon R dans le graphe de Cayley de Z^2, contre l'accepteur.
use std::collections::{HashSet, VecDeque};

#[allow(dead_code)]
fn rjust(s: &str, w: usize) -> String {
    let len = s.chars().count();
    if len >= w {
        s.to_string()
    } else {
        format!("{}{}", " ".repeat(w - len), s)
    }
}

#[allow(dead_code)]
fn bfs_ball_z2(r: i64) -> usize {
    let mut seen: HashSet<(i64, i64)> = HashSet::new();
    seen.insert((0, 0));
    let mut q: VecDeque<(i64, i64, i64)> = VecDeque::new();
    q.push_back((0, 0, 0));
    while let Some((x, y, d)) = q.pop_front() {
        if d == r {
            continue;
        }
        for (dx, dy) in [(1i64, 0i64), (-1i64, 0i64), (0i64, 1i64), (0i64, -1i64)] {
            let n = (x + dx, y + dy);
            if seen.insert(n) {
                q.push_back((n.0, n.1, d + 1));
            }
        }
    }
    seen.len()
}

// Un mot geodesique w_R = a^R de longueur R : le BFS explore la boule, l'accepteur lit le mot.
let rayons: [i64; 8] = [1, 2, 3, 5, 8, 12, 20, 30];
println!("   R  |  BFS noeuds  |  accepteur etats  |  rapport BFS/accepteur");
println!("      |              |                   |{}", " ".repeat(8));
for r in rayons.iter().copied() {
    let bfs_cost = bfs_ball_z2(r); // ~ 2R^2 + 2R + 1  (quadratique)
    let acc_cost = r + 1; // |w_R| + 1 etats parcourus (lineaire)
    println!(
        "  {}  |  {}  |  {}  |  {} x",
        rjust(&r.to_string(), 3),
        rjust(&bfs_cost.to_string(), 9),
        rjust(&acc_cost.to_string(), 15),
        rjust(&format!("{:.1}", bfs_cost as f64 / acc_cost as f64), 8)
    );
}
println!("\nLe BFS croit en ~2R^2 (quadratique), l'accepteur en R (linéaire) : le rapport diverge en ~2R.");


   R  |  BFS noeuds  |  accepteur etats  |  rapport BFS/accepteur


      |              |                   |        


    1  |          5  |                2  |       2.5 x


    2  |         13  |                3  |       4.3 x


    3  |         25  |                4  |       6.2 x


    5  |         61  |                6  |      10.2 x


    8  |        145  |                9  |      16.1 x


   12  |        313  |               13  |      24.1 x


   20  |        841  |               21  |      40.0 x


   30  |       1861  |               31  |      60.0 x


Le BFS croit en ~2R^2 (quadratique), l'accepteur en R (linéaire) : le rapport diverge en ~2R.


### Lecture du résultat

Le BFS visite `2R² + 2R + 1` noeuds pour couvrir la boule de rayon `R` (d'où le `~2R²`), tandis que
l'accepteur ne parcourt que `R + 1` états. Le rapport `BFS/accepteur` croît donc **linéairement avec R**
(colonne de droite) : à `R = 30` le BFS explore ~60 fois plus d'objets que l'automate n'en lit. C'est la
frontière « décider vs explorer » faite chiffre — la structure du groupe rend l'exploration *inutile*.

## 5. La croissance — polynôme (Z²) vs exponentielle (F₂), **comptée**

La signature d'un groupe automatique se lit dans la **croissance** de sa boule. On ne *cite* pas les
formules : on **compte** les sphères et les boules par BFS sur chaque graphe de Cayley, pour `k = 0..8`.

In [6]:

// Comptage des spheres et des boules : Z^2 (polynomial) contre F_2 (exponentiel).
// Lettres encodees : 0 = 'a', 1 = 'a-1', 2 = 'b', 3 = 'b-1' ; l'inverse est t ^ 1.
#[allow(dead_code)]
fn inv_idx(t: u8) -> u8 {
    t ^ 1
}

#[allow(dead_code)]
fn rjust(s: &str, w: usize) -> String {
    let len = s.chars().count();
    if len >= w {
        s.to_string()
    } else {
        format!("{}{}", " ".repeat(w - len), s)
    }
}

#[allow(dead_code)]
fn sphere_z2(k: i64) -> i64 {
    let mut n: i64 = 0;
    for x in -k..=k {
        let ry = k - x.abs();
        n += if ry == 0 { 1 } else { 2 };
    }
    n
}

#[allow(dead_code)]
fn ball_z2(k: i64) -> i64 {
    (0..=k).map(|j| sphere_z2(j)).sum()
}

#[allow(dead_code)]
fn sphere_f2(k: i64) -> i64 {
    let mut level: Vec<Vec<u8>> = vec![Vec::new()];
    for _ in 0..k {
        let mut nxt: Vec<Vec<u8>> = Vec::new();
        for w in &level {
            let last = w.last().copied();
            for t in 0u8..4 {
                if let Some(l) = last {
                    if t == inv_idx(l) {
                        continue;
                    }
                }
                let mut nw = w.clone();
                nw.push(t);
                nxt.push(nw);
            }
        }
        level = nxt;
    }
    level.len() as i64
}

#[allow(dead_code)]
fn ball_f2(k: i64) -> i64 {
    (0..=k).map(|j| sphere_f2(j)).sum()
}

println!("  k  |  Z^2 sphere |  Z^2 boule  |   F2 sphere  |   F2 boule");
println!("     |            |             |              |");
for k in 0..9 {
    println!(
        "  {}  |  {}  |  {}  |  {}  |  {}",
        rjust(&k.to_string(), 2),
        rjust(&sphere_z2(k).to_string(), 8),
        rjust(&ball_z2(k).to_string(), 8),
        rjust(&sphere_f2(k).to_string(), 10),
        rjust(&ball_f2(k).to_string(), 10)
    );
}

// La signature : Z^2 quadratique (2e difference finie constante), F_2 exponentielle (ratio ~3).
assert_eq!(ball_z2(10), 2 * 10i64.pow(2) + 2 * 10 + 1); // 221 : croissance polynomiale (degre 2)
assert_eq!(ball_f2(5), 2 * 3i64.pow(5) - 1); // 485 : croissance exponentielle (base 3)
println!("\nZ^2 : boule = 2k^2 + 2k + 1  (polynome, degre 2).   F_2 : boule = 2\u{b7}3^k - 1  (exponentielle, base 3).");


  k  |  Z^2 sphere |  Z^2 boule  |   F2 sphere  |   F2 boule


     |            |             |              |


   0  |         1  |         1  |           1  |           1


   1  |         4  |         5  |           4  |           5


   2  |         8  |        13  |          12  |          17


   3  |        12  |        25  |          36  |          53


... (2 outputs left out)

   6  |        24  |        85  |         972  |        1457


   7  |        28  |       113  |        2916  |        4373


   8  |        32  |       145  |        8748  |       13121


Z^2 : boule = 2k^2 + 2k + 1  (polynome, degre 2).   F_2 : boule = 2·3^k - 1  (exponentielle, base 3).


### Lecture du résultat

La table fait la différence sans l'affirmer : la boule de `Z^2` suit `2k² + 2k + 1` (les *deuxièmes
différences finies* sont constantes — signature d'une croissance **polynomiale de degré 2**), alors que
celle de `F_2` suit `2·3^k − 1` (le *ratio* d'une boule à la précédente tend vers 3 — signature d'une
croissance **exponentielle**). C'est exactement le contraste que la théorie des groupes automatiques met
en scène : `Z^2` est automatique à croissance polynomiale, `F_2` est automatique à croissance
exponentielle — et les deux admettent un automate fini de mots géodésiques.

### Ce que Rust apporte (mesure)

Rust ne change **rien au résultat** : les deux carnets affichent les mêmes
sorties, caractère pour caractère — l'organe de conformité différentielle le
vérifie, tolérance zéro, cellule par cellule. Ce qui change, c'est le **coût**
du même calcul.

**Protocole.** Les sources **committées** des cellules de calcul (3, 6, 9, 11 et
14) sont extraites des deux carnets en programmes autonomes, exécutés hors du
noyau. La sortie standard de chaque programme est ensuite confrontée à la sortie
committée du carnet correspondant : elle lui est **identique, octet pour octet**
(2 053 octets des deux côtés). C'est cette identité qui autorise à attribuer au
programme le temps mesuré — sans elle, on ne mesurerait qu'un programme
quelconque. Les deux programmes sont ensuite lancés dans la **même campagne** :
sept exécutions de part et d'autre, entrelacées, l'horloge placée *à l'intérieur*
du corps du programme (`Instant::now` en Rust, `time.perf_counter` en Python),
son résultat écrit sur `stderr` — la sortie standard, elle, reste celle du carnet.

| Temps du corps du programme | Python | Rust | Écart |
| --- | --- | --- | --- |
| minimum | 8,585 ms | 5,517 ms | ×1,56 |
| médiane | 8,721 ms | 5,731 ms | ×1,52 |

Commande de mesure : `rustc -O -o search13_timed.exe search13_timed.rs &&
./search13_timed.exe` (rustc 1.98.1, Windows 11) et `python search13_timed.py`
(CPython 3.11.9) ; bibliothèque standard seule, aucun GPU.

**Ce qu'il faut y lire.** L'écart est réel mais **modeste** — un facteur 1,5 —
et c'est la juste mesure d'un calcul petit : automates de six états, boules de
quelques dizaines de nœuds. La médiane est retenue plutôt que la moyenne parce
qu'une exécution Python à 12,903 ms s'écarte du lot (ordonnanceur de la
machine) ; l'écart se lit sur les rangs, pas sur les extrêmes.

Ce que le portage **ne** change pas : la complexité. Les deux colonnes affichent
les mêmes nombres, à l'identique — `Θ(\|mot\|)` côté lecture d'automate,
croissance polynomiale pour `Z^2` et exponentielle pour `F_2`, comme dans le
carnet Python. Ce que Rust change, c'est le **coût constant** de l'exécution.
C'est le résultat attendu d'un jumeau : la valeur du portage se lit sur le
volume, jamais sur la correction.


## Synthèse — où se lit la frontière « décider / explorer »

| Question | Sans structure (Search-02/03) | Avec l'automate du groupe (ce carnet) |
|---|---|---|
| « ce mot est-il géodésique ? » | BFS jusqu'au rayon \|mot\| — **Θ(\|mot\|²)** noeuds | lecture gauche-droite du DFA — **O(\|mot\|)**, mémoire constante |
| Coût en croissance | explore la boule (polynôme pour Z², exponentiel pour F₂) | indépendant de la boule — linéaire en le mot |

Quand le groupe *fournit* l'automate, la géométrie qui coûtait une exploration entière se réduit à un
test d'appartenance à un langage régulier. C'est le sens du « word processing » : le groupe fait le travail
de l'heuristique.

## Exercices

In [7]:
// Exercice 1 — L'accepteur geodesique de Z^3
#[allow(dead_code)]
fn geodesic_accepteur_z3() -> Option<usize> {
    // TODO etudiant
    None // TODO etudiant
}
let _ = geodesic_accepteur_z3;


In [8]:
// Exercice 2 — Le ratio BFS / accepteur sur F_2
#[allow(dead_code)]
fn ratio_bfs_accepteur_f2(_r: i64) -> Option<f64> {
    // TODO etudiant
    None // TODO etudiant
}
let _ = ratio_bfs_accepteur_f2;


In [9]:
// Exercice 3 — Le probleme du mot COMPLET de Z^2 n'est pas regulier
#[allow(dead_code)]
fn pourquoi_pas_regulier() -> Option<String> {
    // TODO etudiant
    None // TODO etudiant
}
let _ = pourquoi_pas_regulier;


## À retenir

- Le langage des mots géodésiques de `Z^2` est **régulier** ; l'automate qui le reconnaît a **6 états**
  et se **construit** (sections 2–3) — il ne se suppose pas.
- Décider « géodésique ? » par cet automate coûte **O(|mot|)** ; le BFS explore **Θ(|mot|²)** — l'écart
  diverge (section 4).
- `Z^2` croît **polynomialement** (`2k²+2k+1`), `F_2` **exponentiellement** (`2·3^k−1`) — compté, pas cité
  (section 5).
- La structure algébrique du groupe, quand elle fournit l'automate, *remplace* l'exploration par une
  décision : c'est le fil qui relie ce carnet à Search-02/03.
- Ce jumeau est écrit en Rust **bibliothèque standard seule** : le même raisonnement, sans dépendance
  d'exécution, et des sorties identiques au caractère près à celles du jumeau Python.

## Bibliographie

- D. Epstein, J. W. Cannon, D. D. Holt, S. Levy, K. C. Paterson, W. P. Thurston, *Word Processing in
  Groups*, 1992 — le « word processing » et les cônes géodésiques.
- M. Gromov, *Groups that act on hyperbolic spaces*, Invent. Math. 1987 — structures automatiques.
- Carnets frères : [Search-02 — Recherche non informée](Search-02-Uninformed.ipynb) (BFS) ·
  [Search-03 — Recherche informée](Search-03-Informed.ipynb) (A*) ·
  [jumeau Python de ce carnet](Search-13-AutomaticGroups-WordProblem.ipynb).